In [ ]:
"""
Purpose:    Diagnostic checks for the novelty-uncertainty mechanism.
            This script examines whether the CV-based uncertainty measure is driven
            by small mean patent values, low dispersion, small cell sizes, or outliers.

Inputs:     data/regression_sample.csv
            data/uncertainty_cell_stats.csv

Outputs:    output/diagnostic_cell_correlations.csv
            output/diagnostic_summary.txt
            output/fig_cv_vs_mean_value.png
            output/fig_cv_vs_std_value.png
            output/fig_novelty_distribution.png
            output/fig_novelty_vs_cv.png

Key Steps:  Load analysis data -> inspect proxy components -> export diagnostics.
How to Run: Open this notebook from Part2/ or Part2/code/ and run all cells.
"""

import os
import pandas as pd
import matplotlib.pyplot as plt


# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------

BASE_DIR = os.path.abspath(".." if os.path.basename(os.getcwd()) == "code" else ".")
DATA_DIR = f"{BASE_DIR}/data"
OUTPUT_DIR = f"{BASE_DIR}/output"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------

def load_diagnostic_data():
    """Load patent-level regression sample and CPC subclass-year cell stats.

    If mean_novelty is missing from cell_stats, construct it from the
    patent-level regression sample and merge it into cell_stats.
    """

    regression_sample_path = f"{DATA_DIR}/regression_sample.csv"
    cell_stats_path = f"{DATA_DIR}/uncertainty_cell_stats.csv"

    print("=" * 70)
    print("Checking input files")
    print("=" * 70)
    print(f"Regression sample path: {regression_sample_path}")
    print(f"Exists: {os.path.exists(regression_sample_path)}")
    print(f"Cell stats path:        {cell_stats_path}")
    print(f"Exists: {os.path.exists(cell_stats_path)}")

    if not os.path.exists(regression_sample_path):
        raise FileNotFoundError(
            f"Cannot find regression_sample.csv at: {regression_sample_path}"
        )

    if not os.path.exists(cell_stats_path):
        raise FileNotFoundError(
            f"Cannot find uncertainty_cell_stats.csv at: {cell_stats_path}"
        )

    regression_sample = pd.read_csv(regression_sample_path)
    cell_stats = pd.read_csv(cell_stats_path)

    # Add mean_novelty if it was not saved in uncertainty_cell_stats.csv
    if "mean_novelty" not in cell_stats.columns:
        mean_novelty_by_cell = (
            regression_sample
            .groupby(["cpc_subclass", "grant_year"])["novelty"]
            .mean()
            .reset_index(name="mean_novelty")
        )

        cell_stats = cell_stats.merge(
            mean_novelty_by_cell,
            on=["cpc_subclass", "grant_year"],
            how="left"
        )

        print("[INFO] mean_novelty was missing, so I constructed it from regression_sample.")

    print("\n" + "=" * 70)
    print("Loaded diagnostic data")
    print("=" * 70)
    print(f"Patent-level regression sample: {len(regression_sample):,} patents")
    print(f"CPC subclass-year cells:        {len(cell_stats):,} cells")
    print("\nCell stats columns:")
    print(list(cell_stats.columns))

    return regression_sample, cell_stats

# ---------------------------------------------------------------------------
# 1. Check novelty construction
# ---------------------------------------------------------------------------

def check_novelty_construction(regression_sample):
    """
    Check whether novelty is correctly constructed from backward similarity.
    Since novelty = - z-score(bsim5), novelty should be negatively correlated with bsim5.
    """

    print("\n" + "=" * 70)
    print("CHECK 1: Novelty construction")
    print("=" * 70)

    corr_bsim_novelty = (
        regression_sample[["bsim5", "novelty"]]
        .corr()
        .loc["bsim5", "novelty"]
    )

    print(f"Correlation between bsim5 and novelty: {corr_bsim_novelty:.4f}")

    if corr_bsim_novelty < -0.95:
        print("[PASS] Novelty is correctly negatively related to backward similarity.")
    else:
        print("[WARN] Novelty may not be correctly constructed. Check the sign.")

    print("\nNovelty summary:")
    print(regression_sample["novelty"].describe())

    plt.figure()
    regression_sample["novelty"].hist(bins=50)
    plt.xlabel("Novelty")
    plt.ylabel("Number of patents")
    plt.title("Distribution of Novelty")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/fig_novelty_distribution.png", dpi=300)
    plt.close()


# ---------------------------------------------------------------------------
# 2. Check whether CV is mechanically driven by mean or SD
# ---------------------------------------------------------------------------

def check_cv_components(cell_stats):
    """
    Since CV = standard deviation / mean, check whether high CV is driven by:
    1. high standard deviation, or
    2. mechanically low mean values.
    """

    print("\n" + "=" * 70)
    print("CHECK 2: CV components")
    print("=" * 70)

    valid_cells = cell_stats.dropna(subset=["tech_uncertainty"]).copy()

    variables = ["tech_uncertainty", "mean_value", "std_value", "patent_count"]
    corr_table = valid_cells[variables].corr()

    print("\nCorrelation among CV, mean value, SD value, and cell size:")
    print(corr_table.round(4))

    corr_table.to_csv(f"{OUTPUT_DIR}/diagnostic_cell_correlations.csv")

    corr_cv_mean = corr_table.loc["tech_uncertainty", "mean_value"]
    corr_cv_std = corr_table.loc["tech_uncertainty", "std_value"]

    print("\nKey interpretation:")
    print(f"Corr(CV, mean value) = {corr_cv_mean:.4f}")
    print(f"Corr(CV, SD value)   = {corr_cv_std:.4f}")

    if corr_cv_mean < -0.5:
        print("[WARN] CV is strongly negatively correlated with mean value.")
        print("       This suggests CV may be mechanically driven by small denominators.")
    else:
        print("[OK] CV is not strongly negatively driven by mean value.")

    if corr_cv_std > 0.5:
        print("[OK] CV is strongly related to value dispersion.")
    else:
        print("[WARN] CV is not strongly related to SD; it may not cleanly capture dispersion.")

    plt.figure()
    plt.scatter(valid_cells["mean_value"], valid_cells["tech_uncertainty"], alpha=0.4)
    plt.xlabel("Mean patent value within CPC subclass-year")
    plt.ylabel("Technology uncertainty: CV")
    plt.title("CV vs. Mean Patent Value")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/fig_cv_vs_mean_value.png", dpi=300)
    plt.close()

    plt.figure()
    plt.scatter(valid_cells["std_value"], valid_cells["tech_uncertainty"], alpha=0.4)
    plt.xlabel("SD of patent value within CPC subclass-year")
    plt.ylabel("Technology uncertainty: CV")
    plt.title("CV vs. SD Patent Value")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/fig_cv_vs_std_value.png", dpi=300)
    plt.close()


# ---------------------------------------------------------------------------
# 3. Check why first stage may be negative
# ---------------------------------------------------------------------------

def check_negative_first_stage(cell_stats):
    """
    Check whether novel technology areas have lower mean value, lower dispersion,
    and therefore lower CV.
    """

    print("\n" + "=" * 70)
    print("CHECK 3: Why is the first stage negative?")
    print("=" * 70)

    valid_cells = cell_stats.dropna(subset=["tech_uncertainty"]).copy()

    if "mean_novelty" not in valid_cells.columns:
        raise ValueError(
            "mean_novelty is missing from uncertainty_cell_stats.csv. "
            "Please add mean_novelty in add_tech_uncertainty() and rerun main.py."
        )

    variables = [
        "mean_novelty",
        "mean_value",
        "std_value",
        "tech_uncertainty",
        "patent_count",
    ]

    novelty_corr = valid_cells[variables].corr()

    print("\nCorrelation between mean novelty and cell-level outcomes:")
    print(novelty_corr.loc["mean_novelty"].round(4))

    corr_novelty_mean = novelty_corr.loc["mean_novelty", "mean_value"]
    corr_novelty_std = novelty_corr.loc["mean_novelty", "std_value"]
    corr_novelty_cv = novelty_corr.loc["mean_novelty", "tech_uncertainty"]

    print("\nKey interpretation:")
    print(f"Corr(mean novelty, mean value) = {corr_novelty_mean:.4f}")
    print(f"Corr(mean novelty, SD value)   = {corr_novelty_std:.4f}")
    print(f"Corr(mean novelty, CV)         = {corr_novelty_cv:.4f}")

    if corr_novelty_cv < 0:
        print("[FINDING] More novel technology cells are associated with lower CV.")
        print("          This helps explain the negative first-stage result.")
    else:
        print("[FINDING] More novel technology cells are not negatively associated with CV.")

    if corr_novelty_std < 0:
        print("[POSSIBLE EXPLANATION] Novel technology cells may be more homogeneous.")
        print("                       Their realized patent values are less dispersed.")
    else:
        print("[NOTE] Novel technology cells do not appear to have lower absolute dispersion.")

    plt.figure()
    plt.scatter(valid_cells["mean_novelty"], valid_cells["tech_uncertainty"], alpha=0.4)
    plt.xlabel("Mean novelty within CPC subclass-year")
    plt.ylabel("Technology uncertainty: CV")
    plt.title("Mean Novelty vs. CV")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/fig_novelty_vs_cv.png", dpi=300)
    plt.close()


# ---------------------------------------------------------------------------
# 4. Check cell size and CV outliers
# ---------------------------------------------------------------------------

def check_cell_size_and_outliers(cell_stats):
    """
    Check whether CV is sensitive to small cells and extreme values.
    """

    print("\n" + "=" * 70)
    print("CHECK 4: Cell size and CV outliers")
    print("=" * 70)

    valid_cells = cell_stats.dropna(subset=["tech_uncertainty"]).copy()

    print("\nPatent count per CPC subclass-year cell:")
    print(valid_cells["patent_count"].describe())

    print("\nCV distribution:")
    print(
        valid_cells["tech_uncertainty"].describe(
            percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
        )
    )

    small_cell_share = (valid_cells["patent_count"] < 20).mean()
    print(f"\nShare of valid cells with fewer than 20 patents: {small_cell_share:.2%}")

    cv_p99 = valid_cells["tech_uncertainty"].quantile(0.99)
    cv_max = valid_cells["tech_uncertainty"].max()

    print(f"99th percentile of CV: {cv_p99:.4f}")
    print(f"Maximum CV:            {cv_max:.4f}")

    if cv_max > 3 * cv_p99:
        print("[WARN] CV has very large extreme values.")
        print("       Winsorizing CV may be important for robustness.")
    else:
        print("[OK] CV extreme values do not look excessively large relative to p99.")


# ---------------------------------------------------------------------------
# 5. Create a written summary
# ---------------------------------------------------------------------------

def write_summary(regression_sample, cell_stats):
    """Save a simple text summary of key diagnostics."""

    valid_cells = cell_stats.dropna(subset=["tech_uncertainty"]).copy()

    corr_bsim_novelty = (
        regression_sample[["bsim5", "novelty"]]
        .corr()
        .loc["bsim5", "novelty"]
    )

    corr_cv_mean = (
        valid_cells[["tech_uncertainty", "mean_value"]]
        .corr()
        .loc["tech_uncertainty", "mean_value"]
    )

    corr_cv_std = (
        valid_cells[["tech_uncertainty", "std_value"]]
        .corr()
        .loc["tech_uncertainty", "std_value"]
    )

    lines = []
    lines.append("Diagnostic Summary")
    lines.append("=" * 70)
    lines.append(f"Patent-level regression sample: {len(regression_sample):,}")
    lines.append(f"Valid CPC subclass-year cells:  {len(valid_cells):,}")
    lines.append("")
    lines.append("1. Novelty construction")
    lines.append(f"   Corr(bsim5, novelty) = {corr_bsim_novelty:.4f}")
    lines.append("   Expected: negative, because novelty is constructed as negative backward similarity.")
    lines.append("")
    lines.append("2. CV components")
    lines.append(f"   Corr(CV, mean patent value) = {corr_cv_mean:.4f}")
    lines.append(f"   Corr(CV, SD patent value)   = {corr_cv_std:.4f}")
    lines.append("   If Corr(CV, mean) is strongly negative, CV may be mechanically driven by small means.")
    lines.append("   If Corr(CV, SD) is weak, CV may not cleanly capture dispersion.")

    if "mean_novelty" in valid_cells.columns:
        corr_novelty_cv = (
            valid_cells[["mean_novelty", "tech_uncertainty"]]
            .corr()
            .loc["mean_novelty", "tech_uncertainty"]
        )

        corr_novelty_mean = (
            valid_cells[["mean_novelty", "mean_value"]]
            .corr()
            .loc["mean_novelty", "mean_value"]
        )

        corr_novelty_std = (
            valid_cells[["mean_novelty", "std_value"]]
            .corr()
            .loc["mean_novelty", "std_value"]
        )

        lines.append("")
        lines.append("3. Negative first-stage diagnosis")
        lines.append(f"   Corr(mean novelty, CV)         = {corr_novelty_cv:.4f}")
        lines.append(f"   Corr(mean novelty, mean value) = {corr_novelty_mean:.4f}")
        lines.append(f"   Corr(mean novelty, SD value)   = {corr_novelty_std:.4f}")
        lines.append("   These correlations help explain why novelty may not predict higher uncertainty.")

    summary_path = f"{OUTPUT_DIR}/diagnostic_summary.txt"

    with open(summary_path, "w") as f:
        f.write("\n".join(lines))

    print(f"\nSaved: {summary_path}")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    regression_sample, cell_stats = load_diagnostic_data()

    check_novelty_construction(regression_sample)
    check_cv_components(cell_stats)
    check_negative_first_stage(cell_stats)
    check_cell_size_and_outliers(cell_stats)
    write_summary(regression_sample, cell_stats)

    print("\n" + "=" * 70)
    print("Diagnostic checks complete.")
    print("=" * 70)
    print(f"Saved figures and tables to: {OUTPUT_DIR}/")


if __name__ == "__main__":
    main()